## Importing the Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# BLOOD CELLS MODEL

# Data preprocessing

TRAINING SET

In [ ]:
blood_train_datagen = ImageDataGenerator(rescale=1./255,shear_range=0.2,zoom_range=0.2,horizontal_flip=True,rotation_range=20, width_shift_range=0.1, height_shift_range=0.1, fill_mode='nearest')

blood_train_set = blood_train_datagen.flow_from_directory('data/Blood Cells/train', target_size=(224, 224), batch_size=32, class_mode='categorical', shuffle=True)

TEST SET

In [ ]:
blood_test_datagen = ImageDataGenerator(rescale=1./255)
blood_test_set = blood_test_datagen.flow_from_directory('data/Blood Cells/test', target_size=(224, 224), batch_size=32, class_mode='categorical', shuffle=False)

In [ ]:
import os

data_path = 'data/Blood cells'
classes = [d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))]

print("True Class Distribution (Recursive):")
total_images = 0
counts = {}

for cls in classes:
    count = sum([len(files) for r, d, files in os.walk(os.path.join(data_path, cls))])
    counts[cls] = count
    total_images += count

for cls, count in counts.items():
    percentage = (count / total_images) * 100 if total_images > 0 else 0
    print(f"{cls}: {count} images ({percentage:.2f}%)")

# Building the TL

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization


NUM_CLASSES = blood_train_set.num_classes  

base_model = MobileNetV2( input_shape=(224,224) + (3,), include_top=False, weights='imagenet')
base_model.trainable = False  

inputs = layers.Input(shape=(224,224) + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

cells_tl_model = models.Model(inputs, outputs)

In [ ]:
print(blood_train_set.class_indices)

# Training the TL

COMPILING THE TL

In [ ]:
cells_tl_model.compile( optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping

callbacks = [
    ReduceLROnPlateau(monitor='val_loss',factor=0.2,patience=4,min_lr=1e-6,verbose=1),
    EarlyStopping(monitor='val_loss',patience=8,restore_best_weights=True,verbose=1)
]

In [ ]:
history_cells_tl = cells_tl_model.fit(blood_train_set,validation_data=blood_test_set,epochs=20, callbacks=callbacks)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-100]:
    layer.trainable = False

callbacks = [
    ReduceLROnPlateau(monitor='val_loss',factor=0.2,patience=3,min_lr=1e-6,verbose=1),
    EarlyStopping(monitor='val_loss',patience=6,restore_best_weights=True,verbose=1)
]

cells_tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history = cells_tl_model.fit( blood_train_set, validation_data=blood_test_set, epochs=10, callbacks=callbacks)

### PLOTTING

In [ ]:
loss, accuracy = cells_tl_model.evaluate(blood_test_set)
print(f"Test loss: {loss}")
print(f"Test accuracy: {accuracy}")

CONFUSION MATRIX

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

blood_test_set.reset()
blood_test_set.shuffle = False 

predictions = cells_tl_model.predict(blood_test_set)
y_pred = np.argmax(predictions, axis=1)

y_true = blood_test_set.classes
class_names = list(blood_test_set.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Blood Classification')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# 6. Detailed Performance Report
print(classification_report(y_true, y_pred, target_names=class_names))

SAVING THE MODEL

In [ ]:
cells_tl_model.save("cells_model.h5")